In [14]:
# Installazione dei pacchetti necessari
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
from dotenv import load_dotenv
import os
import base64
from requests import get, post
import json
import time

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

def get_token():
    auth_string = CLIENT_ID + ":" + CLIENT_SECRET
    auth_bytes = auth_string.encode("utf-8")
    auth_base64 = str(base64.b64encode(auth_bytes), "utf-8")
    
    url = "https://accounts.spotify.com/api/token"
    headers = {
        "Authorization": "Basic " + auth_base64,
        "Content-Type": "application/x-www-form-urlencoded"
    }
    data = {"grant_type": "client_credentials"}
    result = post(url, headers=headers, data=data)
    json_result = result.json()
    return json_result["access_token"]

def get_auth_header(token):
    return {"Authorization": "Bearer " + token}

def get_playlist_tracks(playlist_id, token):
    url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks"
    headers = get_auth_header(token)
    tracks = []
    while url:
        result = get(url, headers=headers)
        json_result = result.json()
        tracks.extend(json_result["items"])
        url = json_result.get("next")
    return tracks

def extract_artists_from_tracks(tracks):
    artists = set()
    for item in tracks:
        track = item["track"]
        for artist in track["artists"]:
            artists.add((artist["id"], artist["name"]))
    return list(artists)

def get_artist_albums(artist_id, token):
    url = f"https://api.spotify.com/v1/artists/{artist_id}/albums?include_groups=album,single&limit=50"
    headers = get_auth_header(token)
    albums = []
    while url:
        result = get(url, headers=headers).json()
        albums.extend(result["items"])
        url = result.get("next")
        time.sleep(0.1)  # per non fare troppi request
    return albums

def get_album_tracks(album_id, token):
    url = f"https://api.spotify.com/v1/albums/{album_id}/tracks?limit=50"
    headers = get_auth_header(token)
    tracks = []
    while url:
        result = get(url, headers=headers).json()
        tracks.extend(result["items"])
        url = result.get("next")
        time.sleep(0.1)
    return tracks

def build_collaboration_graph(playlist_artists, token):
    # Trasformo in dict per riferimento rapido
    artist_id_to_name = {a[0]: a[1] for a in playlist_artists}
    collaborations = {a[1]: set() for a in playlist_artists}

    total_artists = len(playlist_artists)
    for idx, (artist_id, artist_name) in enumerate(playlist_artists, start=1):
        print(f"[{idx}/{total_artists}] Analizzando: {artist_name}...")

        albums = get_artist_albums(artist_id, token)
        for album in albums:
            album_tracks = get_album_tracks(album["id"], token)
            for track in album_tracks:
                track_artists = track["artists"]
                for ta in track_artists:
                    if ta["id"] in artist_id_to_name and ta["id"] != artist_id:
                        collaborations[artist_name].add(artist_id_to_name[ta["id"]])
        time.sleep(0.1)

    # Converte sets in liste per json
    collaborations = {k: list(v) for k, v in collaborations.items()}
    return collaborations

def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

# --- Esecuzione ---
playlist_id = "2azPdjYEz2mGX37sBGQOxG"
token = get_token()
playlist_tracks = get_playlist_tracks(playlist_id, token)
playlist_artists = extract_artists_from_tracks(playlist_tracks)
collaboration_graph = build_collaboration_graph(playlist_artists, token)
save_json(collaboration_graph, "collaboration_graph.json")
print("Grafo di collaborazioni salvato in collaboration_graph.json")


[1/168] Analizzando: DANGERDOOM...
[2/168] Analizzando: Bootsy Collins...
[3/168] Analizzando: Jaydonclover...
[4/168] Analizzando: Terror Reid...
[5/168] Analizzando: rich disease...
[6/168] Analizzando: UZI...
[7/168] Analizzando: Sitcom...
[8/168] Analizzando: Aphex Twin...
[9/168] Analizzando: Pufuleti...
[10/168] Analizzando: David August...
[11/168] Analizzando: Men I Trust...
[12/168] Analizzando: MAVI...
[13/168] Analizzando: A$AP Mob...
[14/168] Analizzando: Earl Sweatshirt...
[15/168] Analizzando: Darrell Cole...
[16/168] Analizzando: Leonard Bernstein...
[17/168] Analizzando: Waldo...
[18/168] Analizzando: brakence...
[19/168] Analizzando: Studio Murena...
[20/168] Analizzando: Paul Kalkbrenner...
[21/168] Analizzando: Action Bronson...
[22/168] Analizzando: Token...
[23/168] Analizzando: Tokyo Tea Room...
[24/168] Analizzando: Simone De Kunovich...
[25/168] Analizzando: Vince Staples...
[26/168] Analizzando: Cristian Vogel...
[27/168] Analizzando: James P. Nichols...
[28/16

JSONDecodeError: Expecting value: line 1 column 1 (char 0)